In [ ]:
import sys

sys.path.append("..")

import os
from datetime import datetime
from glob import glob

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

from nnspike.data import UNetDataset, balance_dataset
from nnspike.models import UNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_label = "img2img"
date_label = datetime.today().strftime("%m%d")

print(f"Train the model on '{device}'")
print(f"Model Label: {model_label}, Date Label: {date_label}")

## Calculate Raw Data Brightness

In [ ]:
label_paths = glob("../storage/labels/*.csv")
label_paths = [path for path in label_paths if os.path.basename(path)[:8] > "20250801"]

df = pd.DataFrame()
for label_path in label_paths:
    label_df = pd.read_csv(label_path)
    df = pd.concat([df, label_df])

df = df[df["use"] == True]

df = df.reset_index(drop=True)  # Reset index for future data balancing
print(f"Total number of training records: {len(df)}")

In [ ]:
brightness = []
indices_to_drop = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing DataFrame"):
    image_path = row["image_path"]
    image = cv2.imread(image_path)
    # Convert to HSV color space
    hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Extract the Value (V) channel
    v_channel = hsv_image[:, :, 2]

    # Calculate the average brightness
    average_brightness = np.mean(v_channel)

    brightness.append(average_brightness)

df["brightness"] = brightness
df.to_csv("../storage/img2img/brightness.csv", index=False)

## Visualize Brightness Distribution

In [ ]:
df = pd.read_csv("../storage/img2img/brightness.csv", dtype={"brightness": float})

plt.figure(figsize=(10, 5))
counts, bins, patches = plt.hist(df["brightness"], bins=20, edgecolor="black")
counts = np.asarray(counts)

counts_threshold = 5000
# Extract data where frequency exceeds the threshold
mask = counts > counts_threshold

if mask.any():
    high_frequency_bins = bins[:-1][mask]  # Get the bin edges
    high_frequency_counts = counts[mask]  # Get the frequencies
    bin_width = bins[1] - bins[0]

    filtered_df = df[
        (df["brightness"] >= high_frequency_bins.min())
        & (df["brightness"] < high_frequency_bins.max() + bin_width)
    ]

    print(
        f"Images trainable counts: {len(filtered_df)} (Threshold: {counts_threshold})"
    )
    df = filtered_df
else:
    print(
        f"No bins exceeded the threshold of {counts_threshold}. Retaining all records."
    )
    filtered_df = df

plt.title("Distribution of ColumnA")
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.show()

## Data Augmentation using Albumentation

In [ ]:
import albumentations as A  # noqa: N812
import cv2

# Define the augmentation pipeline
transform = A.Compose(
    [A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.4, p=1.0)]
)

os.makedirs("../storage/img2img/original", exist_ok=True)
os.makedirs("../storage/img2img/adjusted", exist_ok=True)

# Initialize lists to store image_path and gamma values
original_paths = []
adjusted_paths = []
brightness_df = pd.DataFrame()

for i, row in tqdm(df.iterrows(), total=len(df), desc="Processing DataFrame"):
    image_path = row["image_path"]
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert to RGB

    adjusted_image = transform(image=image)["image"]

    original_path = f"../storage/img2img/original/{i}.png"
    adjusted_path = f"../storage/img2img/adjusted/{i}.png"

    cv2.imwrite(original_path, image)
    cv2.imwrite(adjusted_path, adjusted_image)

    # Save image_path and gamma value
    original_paths.append(original_path)
    adjusted_paths.append(adjusted_path)

# After the loop, add the data back to the DataFrame or save to a new DataFrame
brightness_df["original_paths"] = original_paths
brightness_df["adjusted_paths"] = adjusted_paths

# Optional: Save to CSV
brightness_df.to_csv("../storage/img2img/label.csv", index=False)

## Loading Training Data

In [ ]:
df = pd.read_csv("../storage/img2img/label.csv")

original_paths = df["original_paths"].to_list()
adjusted_paths = df["adjusted_paths"].to_list()

X_all = adjusted_paths
y_all = original_paths

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42
)

train_set = UNetDataset(inputs=X_train, outputs=y_train)
val_set = UNetDataset(inputs=X_val, outputs=y_val)

# Use `inputs, outputs = next(iter(train_loader))` for debugging
train_loader = torch.utils.data.DataLoader(train_set, batch_size=8, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=4, shuffle=True)

In [ ]:
# Loss Function
def reduce_mean(out_im, gt_im):
    return torch.abs(out_im - gt_im).mean()


model = UNet(n_channels=3, n_classes=3)
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)

## Training Loop

In [ ]:
# Initialize TensorBoard writer
writer = SummaryWriter()

# Training loop
num_epochs = 50
best_val_loss = float("inf")
best_model_state = None
best_epoch = 0

# Training loop
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_mode_loss = 0.0
    train_control_loss = 0.0
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        # loss = criterion(outputs, labels
        loss = reduce_mean(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Calculate average losses
    avg_train_loss = train_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0.0
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        with torch.no_grad():
            outputs = model(inputs)
        # loss = criterion(outputs, labels)
        loss = reduce_mean(outputs, labels)
        val_loss += loss.item()

    # Calculate average validation losses
    avg_val_loss = val_loss / len(val_loader)

    # Check if this is the best model so far
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = model.state_dict().copy()  # Deep copy of model state
        best_epoch = epoch + 1
        torch.save(
            model.state_dict(),
            f"../storage/models/{model_label}_{date_label}.pt",
        )
        print(f"New best model found at epoch {best_epoch}!")

    # Log to TensorBoard
    writer.add_scalar("Loss/train_total", avg_train_loss, epoch)
    writer.add_scalar("Loss/val_total", avg_val_loss, epoch)
    writer.add_scalar("Loss/best_val", best_val_loss, epoch)

    print(f"Epoch {epoch + 1}/{num_epochs}:")
    print(f"  Train - Total: {avg_train_loss:.5f}")
    print(f"  Val   - Total: {avg_val_loss:.5f}")

# Load the best model state
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(
        f"\nLoaded best model from epoch {best_epoch} with validation loss: {best_val_loss:.5f}"
    )
else:
    print("\nNo best model found, keeping final model state")

# Close TensorBoard writer
writer.close()

print("\nTraining completed! Feature maps have been logged to TensorBoard.")
print("To view the log, run: tensorboard --logdir=runs")

## Pytorch Model Inference

In [ ]:
import cv2
import numpy as np
import torch

image = cv2.imread("./sample.png", cv2.IMREAD_COLOR)

# Convert BGR to RGB (OpenCV uses BGR by default)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Convert to float and normalize to [0, 1] range
image = image.astype(np.float32) / 255.0

# Convert to PyTorch tensor and rearrange dimensions
# From HWC (Height, Width, Channels) to CHW (Channels, Height, Width)
image_tensor = torch.from_numpy(image).permute(2, 0, 1)

# Add batch dimension: (C, H, W) -> (1, C, H, W)
image_tensor = image_tensor.unsqueeze(0)
image_tensor = image_tensor.to(device)

model.eval()
with torch.no_grad():
    adjusted_image = model(image_tensor)

# Convert back to numpy array for visualization/saving
adjusted_image = adjusted_image.squeeze(0).permute(1, 2, 0).cpu().numpy()

# Convert back to [0, 255] range and uint8
adjusted_image = (adjusted_image * 255).astype(np.uint8)

# Convert RGB back to BGR for OpenCV
adjusted_image = cv2.cvtColor(adjusted_image, cv2.COLOR_RGB2BGR)

## Export to ONNX & Inference

In [ ]:
onnx_path = f"../storage/models/{model_label}_{date_label}.onnx"

model = UNet(n_channels=3, n_classes=3)
model.to(device)
state_dict = torch.load(f"../storage/models/{model_label}_{date_label}.pt")
model.load_state_dict(state_dict)
model.eval()

# Create dummy inputs - adjust dimensions as needed
dummy_image = torch.randn(1, 3, 480, 640).to(device)

# Export to ONNX
torch.onnx.export(
    model,
    (dummy_image,),
    onnx_path,
    export_params=True,
    input_names=["adjusted"],
    output_names=["original"],
)

In [ ]:
import onnxruntime as ort

# Load the ONNX model
onnx_path = f"../storage/models/{model_label}_{date_label}.onnx"
session = ort.InferenceSession(onnx_path)

# Create dummy inputs (same as during export)
dummy_image = np.random.randn(1, 3, 480, 640).astype(np.float32)


# Prepare inputs dictionary
inputs = {"adjusted": dummy_image}

# Run inference
outputs = session.run(["original"], inputs)

# Get results
adjustment = outputs[0][0]

# control_output
# print(f"Control output: {adjustment}")
adjustment.shape

## Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create a figure and a 1x3 grid of subplots
fig, axs = plt.subplots(1, 2, figsize=(14, 4))  # 1 row, 3 columns

# Display each image in its respective subplot
axs[0].imshow(image)
axs[0].set_title("Image 1 (RGB)")
axs[0].axis("off")  # Turn off axes for cleaner image display

axs[1].imshow(adjusted_image)  # Specify colormap for grayscale
axs[1].set_title("Image 2 (Grayscale)")
axs[1].axis("off")

# Adjust layout to prevent overlap
plt.tight_layout()

# Show the plot
plt.show()